# 10 — Unified Projection Scaling Model

**Finite-size scaling surface for distributed residue consistency**

Notebook 09 identified an empirical projection-threshold curve:

```text
link noise ↑ → required projection success ↑
```

Notebook 10 unifies that result across graph sizes:

```text
p_required = f(link_noise, N)
```

The goal is a compact finite-size scaling model:

```text
p_required ≈ A · max(link_noise − noise_crit(N), 0)^β
```

where `N` is graph size.

## Outputs

```text
figures/unified_scaling_fit_by_N.png
figures/noise_crit_vs_inverse_N.png
figures/unified_projection_scaling_surface.png
figures/unified_scaling_residual_check.png

results/unified_scaling_fit_by_N.csv
results/unified_projection_scaling_surface.csv
results/unified_scaling_observed_vs_predicted.csv
results/unified_scaling_summary.json

docs/notebook_10_unified_projection_scaling_model.md
```

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
DOCS_DIR = Path("docs")

for d in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PHASE_LOCK_THRESHOLD = 24 / 25

print("Ready.")
print(f"phase-lock threshold = {PHASE_LOCK_THRESHOLD:.3f}")

## 1. Load Notebook 09 threshold outputs

Primary input:

```text
results/projection_threshold_scaling.csv
```

Expected columns:

```text
n_modules, link_noise, p_required
```

If the CSV is missing, this notebook creates a small demonstration dataset so the workflow remains runnable.

In [ ]:
scaling_path = RESULTS_DIR / "projection_threshold_scaling.csv"

if scaling_path.exists():
    scaling_curve = pd.read_csv(scaling_path)
    print(f"loaded: {scaling_path}")
else:
    print("projection_threshold_scaling.csv not found; using demonstration data.")
    scaling_curve = pd.DataFrame(
        {
            "link_noise": [
                0.00, 0.05, 0.10, 0.15, 0.20,
                0.00, 0.05, 0.10, 0.15,
                0.00, 0.05, 0.10, 0.15,
            ],
            "p_required": [
                0.00, 0.05, 0.60, 0.60, 1.00,
                0.00, 0.10, 0.80, 1.00,
                0.00, 0.00, 0.50, 0.85,
            ],
            "n_modules": [
                12, 12, 12, 12, 12,
                20, 20, 20, 20,
                32, 32, 32, 32,
            ],
        }
    )

required_cols = {"n_modules", "link_noise", "p_required"}
missing = required_cols - set(scaling_curve.columns)

if missing:
    raise ValueError(f"Missing required columns: {missing}")

scaling_curve = scaling_curve.copy()
scaling_curve["n_modules"] = scaling_curve["n_modules"].astype(int)
scaling_curve["link_noise"] = scaling_curve["link_noise"].astype(float)
scaling_curve["p_required"] = scaling_curve["p_required"].astype(float)

print(scaling_curve.head())
print("rows:", len(scaling_curve))
print("N values:", sorted(scaling_curve["n_modules"].unique()))

## 2. Clean threshold data

We fit only valid nonzero, non-saturated points:

```text
0 < p_required < 1
```

This avoids fitting the flat low-noise region and saturation at perfect projection.

In [ ]:
fit_data = scaling_curve.dropna(subset=["p_required"]).copy()

fit_data = fit_data[
    (fit_data["p_required"] > 0)
    & (fit_data["p_required"] < 0.98)
].copy()

print(fit_data)
print("fit rows:", len(fit_data))

## 3. Fit per-size threshold laws

For each graph size:

```text
p_required(N) ≈ A_N · (link_noise − noise_crit,N)^β_N
```

This is an empirical finite-size threshold fit, not a universal law.

In [ ]:
def fit_threshold_power_law(group):
    group = group.sort_values("link_noise").copy()

    positive = group[group["p_required"] > 0]

    if len(positive) == 0:
        return {
            "n_modules": int(group["n_modules"].iloc[0]),
            "noise_crit": np.nan,
            "A": np.nan,
            "beta": np.nan,
            "fit_points": 0,
        }

    noise_crit = float(positive["link_noise"].min())

    fit_group = group[
        (group["link_noise"] > noise_crit)
        & (group["p_required"] > 0)
        & (group["p_required"] < 0.98)
    ].copy()

    if len(fit_group) < 2:
        return {
            "n_modules": int(group["n_modules"].iloc[0]),
            "noise_crit": noise_crit,
            "A": np.nan,
            "beta": np.nan,
            "fit_points": len(fit_group),
        }

    x = fit_group["link_noise"].to_numpy(dtype=float) - noise_crit
    y = fit_group["p_required"].to_numpy(dtype=float)

    eps = 1e-12
    coeffs = np.polyfit(np.log(x + eps), np.log(y + eps), 1)

    beta = float(coeffs[0])
    A = float(np.exp(coeffs[1]))

    return {
        "n_modules": int(group["n_modules"].iloc[0]),
        "noise_crit": noise_crit,
        "A": A,
        "beta": beta,
        "fit_points": len(fit_group),
    }


fit_rows = []

for n_modules, group in scaling_curve.groupby("n_modules"):
    fit_rows.append(fit_threshold_power_law(group))

fit_by_N = pd.DataFrame(fit_rows)
fit_by_N_path = RESULTS_DIR / "unified_scaling_fit_by_N.csv"
fit_by_N.to_csv(fit_by_N_path, index=False)

print(fit_by_N)
print(f"saved: {fit_by_N_path}")

In [ ]:
plt.figure(figsize=(8.8, 5.4))

for n_modules, group in scaling_curve.groupby("n_modules"):
    group = group.dropna(subset=["p_required"]).sort_values("link_noise")

    plt.scatter(
        group["link_noise"],
        group["p_required"],
        s=60,
        label=f"N={n_modules} observed",
    )

    row = fit_by_N[fit_by_N["n_modules"] == n_modules].iloc[0]

    if not np.isnan(row["A"]) and not np.isnan(row["beta"]):
        noise_crit = float(row["noise_crit"])
        A = float(row["A"])
        beta = float(row["beta"])

        x_fit = np.linspace(0, max(group["link_noise"].max() - noise_crit, 1e-6), 200)
        y_fit = A * (x_fit ** beta)
        y_fit = np.clip(y_fit, 0, 1)

        plt.plot(
            noise_crit + x_fit,
            y_fit,
            linewidth=2,
            linestyle="--",
            label=f"N={n_modules} fit β={beta:.2f}",
        )

plt.xlabel("link noise")
plt.ylabel("required projection success")
plt.ylim(-0.02, 1.05)
plt.title("Per-size projection threshold fits")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()

fit_by_N_fig = FIG_DIR / "unified_scaling_fit_by_N.png"
plt.savefig(fit_by_N_fig, dpi=180, bbox_inches="tight")
plt.show()

print(f"saved: {fit_by_N_fig}")

## 4. Fit critical noise as a function of graph size

Use a simple finite-size relation:

```text
noise_crit(N) ≈ c0 + c1 / N
```

This is intentionally conservative because there are only a few graph sizes.

In [ ]:
crit_df = fit_by_N.dropna(subset=["noise_crit"]).copy()
crit_df["inverse_N"] = 1.0 / crit_df["n_modules"]

if len(crit_df) >= 2:
    c1, c0 = np.polyfit(crit_df["inverse_N"], crit_df["noise_crit"], 1)
    c0 = float(c0)
    c1 = float(c1)
else:
    c0 = float(crit_df["noise_crit"].iloc[0]) if len(crit_df) else 0.0
    c1 = 0.0

crit_df["noise_crit_fit"] = c0 + c1 * crit_df["inverse_N"]

print("noise_crit(N) ≈ c0 + c1/N")
print("c0 =", c0)
print("c1 =", c1)
print(crit_df)

In [ ]:
plt.figure(figsize=(8.2, 5.0))

plt.scatter(
    crit_df["inverse_N"],
    crit_df["noise_crit"],
    s=80,
    label="observed critical noise",
)

if len(crit_df) >= 2:
    inv_grid = np.linspace(
        crit_df["inverse_N"].min(),
        crit_df["inverse_N"].max(),
        200,
    )
    plt.plot(
        inv_grid,
        c0 + c1 * inv_grid,
        linestyle="--",
        linewidth=2,
        label="linear fit in 1/N",
    )

for _, row in crit_df.iterrows():
    plt.annotate(
        f"N={int(row['n_modules'])}",
        (row["inverse_N"], row["noise_crit"]),
        textcoords="offset points",
        xytext=(6, 6),
    )

plt.xlabel("1 / N")
plt.ylabel("critical link noise")
plt.title("Critical noise vs inverse graph size")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

crit_fig = FIG_DIR / "noise_crit_vs_inverse_N.png"
plt.savefig(crit_fig, dpi=180, bbox_inches="tight")
plt.show()

print(f"saved: {crit_fig}")

## 5. Build unified scaling model

Use:

```text
β_global = median(β_N)
A_global = median(A_N)
noise_crit(N) = c0 + c1/N
```

Then:

```text
p_hat(noise, N) = A_global · max(noise − noise_crit(N), 0)^β_global
```

In [ ]:
valid_params = fit_by_N.dropna(subset=["A", "beta"]).copy()

if len(valid_params) == 0:
    A_global = 1.0
    beta_global = 1.0
else:
    A_global = float(valid_params["A"].median())
    beta_global = float(valid_params["beta"].median())

def noise_crit_model(N):
    return float(c0 + c1 / float(N))

def p_required_model(link_noise, N):
    nc = noise_crit_model(N)
    x = max(float(link_noise) - nc, 0.0)
    p = A_global * (x ** beta_global)
    return float(np.clip(p, 0.0, 1.0))

print("A_global =", A_global)
print("beta_global =", beta_global)
print("noise_crit model: c0 + c1/N =", c0, "+", c1, "/ N")

## 6. Unified projection scaling surface

Generate a model surface:

```text
x-axis: link noise
y-axis: graph size
color: required projection success
```

In [ ]:
N_grid = np.arange(
    int(scaling_curve["n_modules"].min()),
    int(scaling_curve["n_modules"].max()) + 1,
)

noise_grid = np.linspace(
    0.0,
    max(0.75, scaling_curve["link_noise"].max()),
    120,
)

surface_rows = []

for N in N_grid:
    for noise in noise_grid:
        surface_rows.append(
            {
                "n_modules": int(N),
                "link_noise": float(noise),
                "p_required_hat": p_required_model(noise, N),
                "noise_crit_hat": noise_crit_model(N),
            }
        )

surface_df = pd.DataFrame(surface_rows)

surface_path = RESULTS_DIR / "unified_projection_scaling_surface.csv"
surface_df.to_csv(surface_path, index=False)

print(surface_df.head())
print(f"saved: {surface_path}")

In [ ]:
surface_grid = surface_df.pivot_table(
    index="n_modules",
    columns="link_noise",
    values="p_required_hat",
    aggfunc="mean",
)

plt.figure(figsize=(9, 5.6))

im = plt.imshow(
    surface_grid.values,
    aspect="auto",
    origin="lower",
    extent=[
        surface_grid.columns.min(),
        surface_grid.columns.max(),
        surface_grid.index.min(),
        surface_grid.index.max(),
    ],
    vmin=0,
    vmax=1,
)

plt.colorbar(im, label="predicted required projection success")
plt.xlabel("link noise")
plt.ylabel("graph size N")
plt.title("Unified projection scaling surface")
plt.tight_layout()

surface_fig = FIG_DIR / "unified_projection_scaling_surface.png"
plt.savefig(surface_fig, dpi=180, bbox_inches="tight")
plt.show()

print(f"saved: {surface_fig}")

## 7. Residual check

Compare observed thresholds against model predictions.

A perfect model would lie on the diagonal.

In [ ]:
obs = scaling_curve.dropna(subset=["p_required"]).copy()
obs["p_predicted"] = obs.apply(
    lambda row: p_required_model(row["link_noise"], row["n_modules"]),
    axis=1,
)

obs_path = RESULTS_DIR / "unified_scaling_observed_vs_predicted.csv"
obs.to_csv(obs_path, index=False)

print(obs.head())
print(f"saved: {obs_path}")

In [ ]:
plt.figure(figsize=(6.2, 6.2))

plt.scatter(
    obs["p_required"],
    obs["p_predicted"],
    s=70,
)

plt.plot([0, 1], [0, 1], linestyle="--", linewidth=2, label="ideal")

plt.xlabel("observed required projection success")
plt.ylabel("predicted required projection success")
plt.title("Unified scaling residual check")
plt.xlim(-0.02, 1.02)
plt.ylim(-0.02, 1.02)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

residual_fig = FIG_DIR / "unified_scaling_residual_check.png"
plt.savefig(residual_fig, dpi=180, bbox_inches="tight")
plt.show()

print(f"saved: {residual_fig}")

## 8. Summary exports

In [ ]:
summary_payload = {
    "notebook": "10_unified_projection_scaling_model.ipynb",
    "phase_lock_threshold": PHASE_LOCK_THRESHOLD,
    "model": "p_required_hat = A_global * max(link_noise - noise_crit(N), 0)^beta_global",
    "A_global": A_global,
    "beta_global": beta_global,
    "noise_crit_model": {
        "form": "c0 + c1/N",
        "c0": c0,
        "c1": c1,
    },
    "core_claim": (
        "The projection threshold curve becomes a finite-size scaling surface "
        "p_required = f(link_noise, N)."
    ),
    "figures": [
        "figures/unified_scaling_fit_by_N.png",
        "figures/noise_crit_vs_inverse_N.png",
        "figures/unified_projection_scaling_surface.png",
        "figures/unified_scaling_residual_check.png",
    ],
    "results": [
        "results/unified_scaling_fit_by_N.csv",
        "results/unified_projection_scaling_surface.csv",
        "results/unified_scaling_observed_vs_predicted.csv",
        "results/unified_scaling_summary.json",
    ],
}

summary_path = RESULTS_DIR / "unified_scaling_summary.json"
summary_path.write_text(
    json.dumps(summary_payload, indent=2),
    encoding="utf-8",
)

doc_lines = [
    "# Notebook 10 — Unified Projection Scaling Model",
    "",
    "**Core claim:** threshold curves form a finite-size scaling surface.",
    "",
    "Model:",
    "",
    "`p_required_hat = A_global * max(link_noise - noise_crit(N), 0)^beta_global`",
    "",
    "Outputs:",
    "",
    "- `figures/unified_scaling_fit_by_N.png`",
    "- `figures/noise_crit_vs_inverse_N.png`",
    "- `figures/unified_projection_scaling_surface.png`",
    "- `figures/unified_scaling_residual_check.png`",
    "",
]

doc_path = DOCS_DIR / "notebook_10_unified_projection_scaling_model.md"
doc_path.write_text("\n".join(doc_lines), encoding="utf-8")

print(json.dumps(summary_payload, indent=2))
print(f"saved: {summary_path}")
print(f"saved: {doc_path}")

## 9. Optional zip/export block for Colab

In [ ]:
import zipfile

zip_path = Path("notebook_10_outputs.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"created: {zip_path}")

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))

## Final interpretation

Notebook 09 showed:

```text
projection success threshold increases with link noise
```

Notebook 10 compresses that into:

```text
p_required = f(link_noise, N)
```

This turns the threshold boundary into a finite-size scaling surface.